In [ ]:
!pip install biopython
#!pip install git+https://github.com/songlab-cal/gpn.git
from transformers import AutoTokenizer, AutoModel
import numpy as np
import torch
import time
import os
from Bio import SeqIO
from more_itertools import batched
from transformers.models.bert.configuration_bert import BertConfig #for DNABERT-2

print(torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.cuda.device_count())

In [ ]:
#for dnaberts
#!pip uninstall -y triton # this can lead to errors in GPUs other than A100

In [ ]:
genes = {}

In [ ]:
from Bio import SeqIO
from pathlib import Path

directory = "/kaggle/input/100-genes"

for filename in Path(directory).glob("*.fasta"):
    sequences = []

    for sequence in SeqIO.parse(filename, "fasta"):
        sequences.append(str(sequence.seq))
        
    genes[f"{str(filename).split('/')[-1].rstrip('.fasta')}"] = sequences

In [ ]:
def generateEmbeddings(modelName, gene, sequences):
    tokenizer = AutoTokenizer.from_pretrained(modelName, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        modelName, trust_remote_code=True,
    ).to(device)
    embeddings = []

    start = time.time()
    print(f"Starting to generate {modelName}'s embeddings.")
    for sequence in sequences:
        inputs = tokenizer(
            sequence,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        embedding = [item for item in outputs.last_hidden_state.cpu().mean(dim=1).squeeze().numpy()]
        embeddings.append(embedding)
    finish = time.time()

    model_dir = os.path.join('kaggle/working', modelName.split('/')[1])
    os.makedirs(model_dir, exist_ok=True)
    
    output_file = os.path.join(model_dir, f"{gene}_embeddings.txt")
    with open(output_file, "w") as f:
        f.write(str(embeddings))
    
    del model
    del tokenizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    print(
        f"Finished generating {modelName}'s embeddings in {round(finish - start)} seconds."
    )

In [ ]:
def generateBatchedEmbeddings(modelName, gene, sequences):
    tokenizer = AutoTokenizer.from_pretrained(modelName, trust_remote_code=True)
    if modelName.split('/')[0] == 'zhihan1996':
        print('Doing a dnabert model ah?')
        config = BertConfig.from_pretrained(modelName) #for DNABERT-2
        model = AutoModel.from_pretrained(
            modelName, trust_remote_code=True, config=config
        ).to(device)
    else:
        model = AutoModel.from_pretrained(
            modelName, trust_remote_code=True
        ).to(device)
    embeddings = []

    start = time.time()
    print(f"Starting to generate {modelName}'s embeddings.")
    batchedSequences = list(batched(sequences, 10))
    
    for batch in batchedSequences:
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=False,
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        result = outputs.last_hidden_state.cpu().mean(dim=1).squeeze().numpy()
        #result = outputs[1].cpu().numpy().squeeze() #For DNABERT-S/2
        
        if result.ndim == 1:
            embedding = [item for item in result]
            embeddings.append(embedding)
        else: #since the embeddings are batched here
            for element in result:
                embedding = [item for item in element]
                embeddings.append(embedding)
                
    finish = time.time()

    model_dir = os.path.join('kaggle/working', modelName.split('/')[1])
    os.makedirs(model_dir, exist_ok=True)
    
    output_file = os.path.join(model_dir, f"{gene}_embeddings.txt")
    with open(output_file, "w") as f:
        f.write(str(embeddings))
        
    del model
    del tokenizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    print(
        f"Finished generating {modelName}'s embeddings in {round(finish - start)} seconds."
    )

In [ ]:
models = [
    #"LongSafari/hyenadna-medium-450k-seqlen-hf",
    #"LongSafari/hyenadna-large-1m-seqlen-hf",
    #"AIRI-Institute/gena-lm-bigbird-base-t2t",
    #"songlab/gpn-brassicales",
    #"InstaDeepAI/nucleotide-transformer-500m-human-ref",
    #"PoetschLab/GROVER",
    "InstaDeepAI/nucleotide-transformer-2.5b-multi-species", #55 genes done
    #"InstaDeepAI/nucleotide-transformer-2.5b-1000g", yet to do

    #"zhihan1996/DNABERT-S",
    #"zhihan1996/DNABERT-2-117M",
]

In [ ]:
#import gpn.model

In [ ]:
for model in models:
    count = 0
    print(f"Now using gLM: {model}")
    for key, value in genes.items():
        count += 1
        generateBatchedEmbeddings(model, key, value)
        print(f"done with {count} genes.")